# Gender Extraction NLP Pilot
This notebook demonstrates a proof-of-concept pipeline over a sample of 100 random snippets.

In [ ]:
import pandas as pd
import spacy
import re
from IPython.display import display

# Load the spaCy model
nlp = spacy.load("en_core_web_sm")

# Load the comprehensive PR/PS snippet dataset
df_all = pd.read_csv('../../data/pr_ps_snippets.csv')
print(f"Total snippets available: {len(df_all)}")

# Sample 100 random snippets
df_sample = df_all.sample(n=min(100, len(df_all)), random_state=42).copy()
print("Initialized 100-snippet pilot subset.")


## Step 1: Define the Hybrid Gender Pipeline

In [ ]:

male_keywords = {'he', 'him', 'his', 'man', 'boy', 'gentleman', 'father', 'brother', 'husband', 'uncle'}
female_keywords = {'she', 'her', 'hers', 'woman', 'girl', 'lady', 'mother', 'sister', 'wife', 'aunt'}
first_person_keywords = {'i', 'me', 'my', 'mine', 'we', 'us', 'our', 'ours'}

def get_contextual_gender(text, keyword):
    # Baseline heuristic: Count pronouns in the 100 word window around the keyword
    words = [w.lower() for w in re.findall(r'\w+', text)]
    
    male_count = sum(1 for w in words if w in male_keywords)
    female_count = sum(1 for w in words if w in female_keywords)
    first_person_count = sum(1 for w in words if w in first_person_keywords)
    
    if first_person_count > (male_count + female_count) and first_person_count > 2:
        return "Unknown (First-Person)"
    if male_count > female_count * 2:
        return "Male (Context)"
    elif female_count > male_count * 2:
        return "Female (Context)"
    elif male_count > female_count:
        return "Male (Context-Weak)"
    elif female_count > male_count:
        return "Female (Context-Weak)"
    return "Unknown/Neutral"

def get_syntactic_gender(text, keyword):
    doc = nlp(text)
    
    # 1. Find the token matching the target keyword
    target_token = None
    for token in doc:
        # Check strict substring
        if keyword.lower() in token.text.lower():
            target_token = token
            break
            
    if not target_token:
        # Keyword might have been stripped of punctuation weirdly, fall back
        return None
        
    # 2. Traverse up the dependency tree to find the head verb/action
    head = target_token.head
    for _ in range(3):
        if head.pos_ == "VERB" or head.dep_ == "ROOT":
            break
        head = head.head
        
    # 3. Look for subjects attached to this action
    for child in head.children:
        if child.dep_ in ["nsubj", "nsubjpass", "poss"]:
            subject_text = child.text.lower()
            if subject_text in male_keywords:
                return "Male (Syntactic)"
            elif subject_text in female_keywords:
                return "Female (Syntactic)"
            elif subject_text in first_person_keywords:
                return "Unknown (First-Person)"
    
    return None

def predict_gender(row):
    syn = get_syntactic_gender(row['Snippet'], row['Keyword'])
    if syn:
        return syn
    
    return get_contextual_gender(row['Snippet'], row['Keyword'])

df_sample['Predicted_Gender'] = df_sample.apply(predict_gender, axis=1)
print("Pipeline Execution Complete!")


## Step 2: Evaluate the Results

In [ ]:
# Display Results
pd.set_option('display.max_colwidth', None)

df_display = df_sample[['Book_ID', 'Title', 'Keyword', 'Predicted_Gender', 'Snippet']]

print("Distribution of predicted genders:")
print(df_sample['Predicted_Gender'].value_counts())

display(df_display.head(20))
